# Stage 1 - Data Aggregation
Loads parquets produced by `Pre_processing.ipynb` and builds a unified event dataset, as well as NLP paragraphs that overview
details of each event in a more narrative manner.

**Inputs:**
- `official_reports.parquet`
- `citizen_reports.parquet`
- `aq_NL49551.parquet` … `aq_NL49553.parquet`
- `postcode_geometry_wkt.parquet`
- `knmi_weather.parquet`

**Outputs (to `./Processed/`):**
- `aggregated_events.parquet` — one row per KGF event, all scores attached
- `matched_citizen_reports.parquet` — one row per matched citizen report
- `event_paragraphs.parquet` — natural language summary + structured JSON per event

**Usage:** Run all cells. If you want to see specific parts, you can do so in the final cell, simply change the desired dat period and chimney if there is a preference. By default, it goes over March for both chimneys.

**References:**
- Pasquill-Gifford stability: Turner (1994), *Workbook of Atmospheric Dispersion Estimates*
- Gaussian puff: Gifford (1976), *Turbulent Diffusion-Typing Schemes*
- Wind profile power law: Irwin (1979), *Theoretical variation of wind profile power-law exponent*
- Coke oven chemical weights: EU BREF Iron & Steel (2022), Tsai et al. (2019) *Atmospheric Environment*

## 0 · Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import math
import json
import warnings
from shapely import wkt
from shapely.geometry import Point
import geopandas as gpd
from pyproj import Transformer

warnings.filterwarnings('ignore')

PROCESSED_DIR = Path("./Processed")

# Chimney coordinates (extracted from https://gisviewer.odnzkg.nl/index.php?@TataSteel)
# NOTE: stack height are defaulted at 35m due to the lack of concrete information regarding the metrics of KGF facilities
CHIMNEYS = {
    'KGF1': {'lat': 52.474106, 'lon': 4.611345, 'stack_height_m': 35},
    'KGF2': {'lat': 52.476475, 'lon': 4.582798, 'stack_height_m': 35},
}

# Monitor coordinates (pulled from KNMI dataset)
MONITORS = {
    'NL49551': {'lat': 52.463040, 'lon': 4.601776},
    'NL49553': {'lat': 52.494000, 'lon': 4.602383},
    'NL49557': {'lat': 52.490052, 'lon': 4.599313},
    'NL49570': {'lat': 52.489271, 'lon': 4.640536},
    'NL49572': {'lat': 52.474402, 'lon': 4.628525},
    'NL49573': {'lat': 52.477999, 'lon': 4.578774},
}
MEETLOCATIES = list(MONITORS.keys())

# convert Dutch postcode polygon shapes from GPS coordinates into a flat metre-based grid
_to_rd   = Transformer.from_crs("EPSG:4326", "EPSG:28992", always_xy=True)
_from_rd = Transformer.from_crs("EPSG:28992", "EPSG:4326", always_xy=True)

def _geom_to_rd(geom):
    # Reproject a Shapely geometry from WGS84 to RD New (EPSG:28992)
    from shapely.ops import transform as shp_transform
    return shp_transform(lambda x, y: _to_rd.transform(x, y), geom)

print("Setup complete.")

In [ ]:
# Pasquill-Gifford stability table
# Daytime rows: (Q_low, Q_high, [(wind_speed_max, class), ...])
# Nighttime rows: (N_low, N_high, [(wind_speed_max, class), ...])
# IMPORTANT INFORMATION: Turner (1994) daytime thresholds are in W/m².
# KNMI global radiation Q is in J/cm²/h. Conversion: 1 J/cm²/h = 2.778 W/m²
PG_DAY = [
    (216, np.inf, [(2,'A'),   (3,'AB'),  (5,'B'),   (np.inf,'C')]),
    (108, 216,    [(2,'AB'),  (3,'B'),   (5,'BC'),  (np.inf,'CD')]),
    (54,  108,    [(2,'B'),   (3,'BC'),  (5,'C'),   (np.inf,'D')]),
    (0,   54,    [(2,'C'),   (3,'CD'),  (5,'D'),   (np.inf,'D')]),
]
PG_NIGHT = [
    (0, 4, [(2,'F'),  (3,'E'),  (5,'D'),  (np.inf,'D')]),
    (4, 9, [(2,'E'),  (3,'E'),  (5,'D'),  (np.inf,'D')]),
]

# Half-angle (degrees) per class
PG_HALFANGLE = {'A':45, 'B':35, 'C':25, 'D':15, 'E':10, 'F':7}

# Briggs (1973) open-country σ_y coefficients: σ_y(x) = a * x^b, x in metres
PG_SIGMA_Y = {
    'A':(0.22,0.894), 'B':(0.16,0.894), 'C':(0.11,0.894),
    'D':(0.08,0.894), 'E':(0.06,0.894), 'F':(0.04,0.894),
}

# Wind profile alpha by class Irwin (1979)
PG_ALPHA = {'A':0.08, 'B':0.09, 'C':0.11, 'D':0.16, 'E':0.32, 'F':0.54}

# Chemical classification based on EU BREF Iron & Steel (2022), EMEP/EEA Air Pollutant Emission Inventory Guidebook
# Fingerprint: primary gas products of coke ovens
# General: non-specific combustion/particulate indicators
COKE_FINGERPRINT_CHEMS   = ['SO2', 'H2S', 'NO', 'NOx']
GENERAL_COMBUSTION_CHEMS = ['BC', 'PM10', 'PM2.5', 'CO']
 
# NO2 tracked individually; it is a secondary oxidation product not directly
# produced by coke oven combustion. Calculated in case it becomes useful.
TARGET_CHEMICALS = COKE_FINGERPRINT_CHEMS + GENERAL_COMBUSTION_CHEMS + ['NO2']
 
# NL49573 is 321 m from KGF2 and should theoretically always be included for KGF2 events regardless of
# wind direction, hence the over-ride
PROXIMITY_OVERRIDE = {('KGF2', 'NL49573')}
 
DISCARD_AFTER_MIN = 90     # approximate cutoff after puff departure to avoid situations where
                           # low wind speed connects reports filed 24+ hours later
 
# 30-day rolling Z-score baseline
ROLLING_DAYS = 30
 
# PG plain-language descriptions for LLM paragraph
PG_PLAIN = {
    'A':  'very unstable — strong convection, rapid plume dispersion',
    'AB': 'unstable to moderately unstable',
    'B':  'moderately unstable — good vertical mixing',
    'BC': 'moderately unstable to neutral',
    'C':  'slightly unstable',
    'CD': 'slightly unstable to neutral',
    'D':  'neutral — moderate dispersion, typical daytime conditions',
    'DE': 'neutral to slightly stable',
    'E':  'slightly stable — reduced dispersion',
    'F':  'stable — nocturnal, plume stays concentrated near ground level, '
          'highest ground-level exposure risk',
} 
print("Constants loaded.")

## 1 · Load processed data

In [ ]:
official_reports = pd.read_parquet(PROCESSED_DIR / "official_reports.parquet")
citizen_reports  = pd.read_parquet(PROCESSED_DIR / "citizen_reports.parquet")
weather_data     = pd.read_parquet(PROCESSED_DIR / "knmi_weather.parquet")

gdf_pc = gpd.read_file(PROCESSED_DIR / "postcode_geometry.gpkg")
gdf_pc = gdf_pc.to_crs("EPSG:4326")  # reproject to WGS84

postcode_zones = pd.DataFrame({
    'postcode':      gdf_pc['postcode'],
    'geometry':      gdf_pc['geometry'].apply(lambda g: g.wkt),
    'geometry_shape': gdf_pc['geometry'],
    'centroid_lat':  gdf_pc['geometry'].centroid.y,
    'centroid_lon':  gdf_pc['geometry'].centroid.x,
})

aq_monitors = {}
for loc in MEETLOCATIES:
    path = PROCESSED_DIR / f"aq_{loc}.parquet"
    if path.exists():
        aq_monitors[loc] = pd.read_parquet(path)

kgf_events = official_reports[
    official_reports['inrichting'].str.upper().str.contains('KGF', na=False)
].copy().reset_index(drop=True)
print(f"KGF events: {len(kgf_events)}")

print(f"Official reports : {len(official_reports)} rows")
print(f"Citizen reports  : {len(citizen_reports)} rows")
print(f"Postcode zones   : {len(postcode_zones)} postcodes")
print(f"Weather rows     : {len(weather_data)}")
print(f"AQ monitors loaded: {list(aq_monitors.keys())}")

In [ ]:
from zoneinfo import ZoneInfo

AMS = ZoneInfo('Europe/Amsterdam')

# Convert a datetime Series to tz-naive Amsterdam local time.
def _to_ams_naive(series, source_tz=None):
    if series.dt.tz is not None:
        return series.dt.tz_convert(AMS).dt.tz_localize(None)

    if source_tz is None:
        # Treat as UTC
        return (series
                .dt.tz_localize('UTC')
                .dt.tz_convert(AMS)
                .dt.tz_localize(None))
    else:
        return (series
                .dt.tz_localize(source_tz, ambiguous='NaT', nonexistent='NaT')
                .dt.tz_convert(AMS)
                .dt.tz_localize(None))

# Weather: stored as UTC-naive
weather_data['datetime_utc'] = _to_ams_naive(weather_data['datetime_utc'], source_tz=None)
weather_data = weather_data.rename(columns={'datetime_utc': 'datetime_local'})

# Citizen reports: stored as Amsterdam-local naive
citizen_reports['datetime_waarneming'] = _to_ams_naive(
    citizen_reports['datetime_waarneming'], source_tz=AMS
)

# AQ monitors: tz-aware UTC+1 (already handled by tz_convert branch)
for loc in aq_monitors:
    aq_monitors[loc]['begindatumtijd'] = _to_ams_naive(
        aq_monitors[loc]['begindatumtijd'], source_tz=None
    )

# Official reports: align timezone.
if kgf_events['datetime_voorval'].dt.tz is not None:
    kgf_events['datetime_voorval'] = _to_ams_naive(kgf_events['datetime_voorval'])
else:
    # Spot-check: event index 0 should have a plausible Amsterdam local hour (0-23).
    sample_hour = kgf_events['datetime_voorval'].iloc[0].hour
    assert 0 <= sample_hour <= 23, (
        f"datetime_voorval sample hour {sample_hour} out of range — "
        "check whether this column needs UTC→Amsterdam conversion."
    )

# Validation
print("weather_data   :", weather_data['datetime_local'].min(), ">", weather_data['datetime_local'].max())
print("citizen_reports:", citizen_reports['datetime_waarneming'].min(), ">", citizen_reports['datetime_waarneming'].max())
for loc in aq_monitors:
    print(f"aq {loc}       :", aq_monitors[loc]['begindatumtijd'].min(), ">", aq_monitors[loc]['begindatumtijd'].max())

## 2 · Additional functions

In [ ]:
# Great-circle distance in metres between two WGS84 coordinates.
def haversine_m(lat1, lon1, lat2, lon2):
    # Mean earth radius
    R = 6_371_000
    φ1, φ2 = math.radians(lat1), math.radians(lat2)
    dφ = math.radians(lat2 - lat1)
    dλ = math.radians(lon2 - lon1)
    a = math.sin(dφ/2)**2 + math.cos(φ1)*math.cos(φ2)*math.sin(dλ/2)**2
    return R * 2 * math.asin(math.sqrt(a))

# Forward azimuth (degrees, 0=N clockwise) from point 1 to point 2.
# Used to compute the bearing from chimney to monitor/postcode centroid,
# which is then compared against the estimated plume direction.
def bearing_deg(lat1, lon1, lat2, lon2):
    φ1, φ2 = math.radians(lat1), math.radians(lat2)
    dλ = math.radians(lon2 - lon1)
    x = math.sin(dλ) * math.cos(φ2)
    y = math.cos(φ1)*math.sin(φ2) - math.sin(φ1)*math.cos(φ2)*math.cos(dλ)
    return (math.degrees(math.atan2(x, y)) + 360) % 360

# Linear interpolation of KNMI hourly observations to event timestamp.
def interpolate_weather(event_dt, df):
    before = df[df['datetime_local'] <= event_dt].tail(1)
    after  = df[df['datetime_local'] >  event_dt].head(1)

    if before.empty or after.empty:
        # Edge case: event outside weather record range — use nearest row
        row = df.iloc[(df['datetime_local'] - event_dt).abs().argsort()[:1]].iloc[0]
        return row.to_dict()

    t0 = before.iloc[0]['datetime_local']
    t1 = after.iloc[0]['datetime_local']

    w  = (event_dt - t0).total_seconds() / (t1 - t0).total_seconds()
    result = {}

    for col in ['wind_dir_deg','wind_speed_ms','temp_c','radiation_jcm2','N','U',
                'pressure_hpa','precip_mm','wind_gust_ms']:
        if col not in before.columns:
            continue
        v0 = pd.to_numeric(before.iloc[0][col], errors='coerce')
        v1 = pd.to_numeric(after.iloc[0][col],  errors='coerce')
        if col == 'wind_dir_deg':
            # Circular interpolation: find shortest angular path [-180, 180]
            diff = ((v1 - v0 + 180) % 360) - 180
            result[col] = (v0 + w * diff) % 360
        else:
            # If either bound is NaN, fall back to the known value
            result[col] = v0 + w*(v1-v0) if not (np.isnan(v0) or np.isnan(v1)) else v0

    return result


def pasquill_class(weather, event_dt):
    """Classify atmospheric stability using the Pasquill-Gifford scheme.

    Day/night split: daytime requires hour 07-19 AND solar radiation > 10 J/cm²
    (the Q>10 guard avoids misclassifying overcast days as daytime stable).
    Daytime stability is driven by solar radiation Q; nighttime by cloud cover N.
    Returns 'D' (neutral) as the safe default if no table row matches —
    consistent with Turner (1994) and the dominant class in the IJmuiden dataset.

    References: Turner (1994), Workbook of Atmospheric Dispersion Estimates.
    """
    ws   = weather.get('wind_speed_ms', 2.0)
    Q    = weather.get('radiation_jcm2', 0.0)
    N    = weather.get('N', 5.0)
    hour = event_dt.hour
    is_day = (7 <= hour <= 19) and Q is not None and not np.isnan(Q) and Q > 10
    table = PG_DAY if is_day else PG_NIGHT
    ref   = Q if is_day else (N if N is not None and not np.isnan(N) else 5.0)
    for (lo, hi, bins) in table:
        if lo <= ref < hi or (hi == np.inf and ref >= lo):
            for (ws_max, cls) in bins:
                if ws < ws_max:
                    return cls
    return 'D'


def resolve_pg(cls):
    """Return (half_angle, sigma_y_a, sigma_y_b) for a PG stability class.

    Transitional classes (e.g. 'AB', 'BC', 'CD') arise from the Turner table
    at boundary conditions. Their dispersion parameters are linearly averaged
    from the two bounding classes, consistent with the interpolation approach
    in Seinfeld & Pandis (2006, §18.2).
    """
    if len(cls) == 1:
        return PG_HALFANGLE[cls], PG_SIGMA_Y[cls][0], PG_SIGMA_Y[cls][1]
    c1, c2 = cls[0], cls[1]
    ha = (PG_HALFANGLE[c1] + PG_HALFANGLE[c2]) / 2
    a  = (PG_SIGMA_Y[c1][0] + PG_SIGMA_Y[c2][0]) / 2
    b  = (PG_SIGMA_Y[c1][1] + PG_SIGMA_Y[c2][1]) / 2
    return ha, a, b

In [ ]:
def puff_times(chimney_key, event_dt, weather, target_lat, target_lon,
               target_geom=None):
    chim       = CHIMNEYS[chimney_key]
    pg         = pasquill_class(weather, event_dt)
    ha, sa, sb = resolve_pg(pg)
    alpha = (PG_ALPHA.get(pg[0], 0.15) + PG_ALPHA.get(pg[-1], 0.15)) / 2
    v_stack    = max(weather.get('wind_speed_ms', 1.0) * (chim['stack_height_m'] / 10) ** alpha, 0.5)
    wind_from  = weather.get('wind_dir_deg', 0)
    plume_dir  = (wind_from + 180) % 360

    dist_m     = haversine_m(chim['lat'], chim['lon'], target_lat, target_lon)
    tgt_bear   = bearing_deg(chim['lat'], chim['lon'], target_lat, target_lon)
    angle_diff = abs(((tgt_bear - plume_dir + 180) % 360) - 180)
    in_path    = angle_diff <= ha

    sigma_y = sa * (dist_m ** sb)
    sigma_x = sigma_y

    if target_geom is not None and in_path:
        try:
            # Reproject to metric CRS before any geometry operations
            chim_pt_rd  = Point(*_to_rd.transform(chim['lon'], chim['lat']))
            geom_rd     = _geom_to_rd(target_geom)

            # Closest point on the polygon boundary facing the chimney
            closest_rd  = geom_rd.exterior.interpolate(
                geom_rd.exterior.project(chim_pt_rd)
            )
            # Farthest vertex (deepest into the plume path)
            farthest_rd = max(
                geom_rd.exterior.coords,
                key=lambda xy: Point(xy).distance(chim_pt_rd)
            )

            # Convert back to WGS84 lat/lon for haversine
            closest_lon,  closest_lat  = _from_rd.transform(closest_rd.x,  closest_rd.y)
            farthest_lon, farthest_lat = _from_rd.transform(*farthest_rd)

            dist_arrival_m   = haversine_m(chim['lat'], chim['lon'],
                                           closest_lat,  closest_lon)
            dist_departure_m = haversine_m(chim['lat'], chim['lon'],
                                           farthest_lat, farthest_lon)
        except Exception:
            dist_arrival_m, dist_departure_m = dist_m * 0.9, dist_m * 1.1
    else:
        dist_arrival_m, dist_departure_m = dist_m, dist_m

    arrival   = event_dt + pd.Timedelta(seconds=dist_arrival_m   / v_stack)
    peak      = event_dt + pd.Timedelta(seconds=dist_m           / v_stack)
    departure = event_dt + pd.Timedelta(seconds=dist_departure_m / v_stack
                                        + sigma_x / v_stack)

    result = {
        'arrival': arrival, 'peak': peak, 'departure': departure,
        'distance_m': dist_m, 'in_path': in_path,
        'puff_speed_ms': v_stack, 'pg_class': pg,
        'half_angle_deg': ha, 'sigma_y_m': sigma_y, 'sigma_x_m': sigma_x,
        'angle_diff_deg': angle_diff,
    }

    MAX_TRAVEL_HOURS = 3
    if (arrival - event_dt).total_seconds() > MAX_TRAVEL_HOURS * 3600:
        result['in_path'] = False   # flag the row instead of deleting it,
        # downstream code can learn to distinguish "wrong direction" from "too far"

    return result

## 3 · Pre-compute rolling z-scores

In [ ]:
def compute_rolling_zscore(df, chemicals=TARGET_CHEMICALS, window_days=ROLLING_DAYS):
    df = df.sort_values('begindatumtijd').reset_index(drop=True)
    window = window_days * 24
    for chem in chemicals:
        if chem not in df.columns:
            df[f'{chem}_z'] = np.nan
            continue
        roll_mean = df[chem].rolling(window, min_periods=72).mean()
        roll_std  = df[chem].rolling(window, min_periods=72).std()
        df[f'{chem}_z'] = (df[chem] - roll_mean) / roll_std.replace(0, np.nan)
    return df

def get_aq_at_peak(monitor_id, peak_time, aq_z_df):
    """
    Return interpolated z-scores and raw concentrations at peak_time.
 
    Returns dict containing:
      - per-chemical z-scores (keyed by chemical name)
      - raw_concentrations: dict of actual µg/m³ readings per chemical
      - fingerprint_z : max(SO2, H2S, NO, NOx)
      - general_z     : max(BC, PM10, PM2.5, CO)
      - severity_z    : max(fingerprint_z, general_z)  — primary signal
      - composite_z   : alias for severity_z
 
    NO2 is interpolated and returned individually but excluded from both
    group maximums (secondary oxidation product, not direct coke emission).
    """
    if aq_z_df is None or aq_z_df.empty:
        return {}
 
    df = aq_z_df
    before = df[df['begindatumtijd'] <= peak_time].tail(1)
    after  = df[df['begindatumtijd'] >  peak_time].head(1)
 
    result = {}
    raw    = {}
    chems_avail = [c for c in TARGET_CHEMICALS if f'{c}_z' in df.columns]
 
    if before.empty:
        rows = [after.iloc[0]] if not after.empty else []
    elif after.empty:
        rows = [before.iloc[0]]
    else:
        t0 = before.iloc[0]['begindatumtijd']
        t1 = after.iloc[0]['begindatumtijd']
        w  = (peak_time - t0).total_seconds() / (t1 - t0).total_seconds()
        for chem in chems_avail:
            v0 = before.iloc[0][f'{chem}_z']
            v1 = after.iloc[0][f'{chem}_z']
            result[chem] = (
                float(v0 + w * (v1 - v0))
                if pd.notna(v0) and pd.notna(v1)
                else (float(v0) if pd.notna(v0) else np.nan)
            )
            if chem in df.columns:
                r0 = before.iloc[0][chem]
                r1 = after.iloc[0][chem]
                raw[chem] = (
                    float(r0 + w * (r1 - r0))
                    if pd.notna(r0) and pd.notna(r1)
                    else (float(r0) if pd.notna(r0) else np.nan)
                )
        rows = []
 
    for row in rows:
        for chem in chems_avail:
            v = row[f'{chem}_z']
            result[chem] = float(v) if pd.notna(v) else np.nan
            if chem in df.columns:
                r = row[chem]
                raw[chem] = float(r) if pd.notna(r) else np.nan
 
    # Maximum sub-index by chemical class
    fp_vals = [result[c] for c in COKE_FINGERPRINT_CHEMS
               if c in result and pd.notna(result[c])]
    gn_vals = [result[c] for c in GENERAL_COMBUSTION_CHEMS
               if c in result and pd.notna(result[c])]
 
    fingerprint_z = float(max(fp_vals)) if fp_vals else np.nan
    general_z     = float(max(gn_vals)) if gn_vals else np.nan
    all_vals      = [v for v in [fingerprint_z, general_z] if pd.notna(v)]
    severity_z    = float(max(all_vals)) if all_vals else np.nan
 
    result['fingerprint_z']      = fingerprint_z
    result['general_z']          = general_z
    result['severity_z']         = severity_z
    result['composite_z']        = severity_z 
    result['raw_concentrations'] = raw
 
    return result


print("Computing rolling z-scores for all monitors...")
aq_with_z = {}
for loc, df in aq_monitors.items():
    aq_with_z[loc] = compute_rolling_zscore(df.copy())
    z_present = [c for c in TARGET_CHEMICALS
                 if f'{c}_z' in aq_with_z[loc].columns
                 and aq_with_z[loc][f'{c}_z'].notna().any()]
    print(f"  {loc}: z-scores for {z_present}")

## 4 · Citizen report scoring function

In [ ]:
def _detail_category(detail_str):
    # Map descriptor string to summary category for paragraph output.
    COKE_SPECIFIC = {'Kooks', 'Zwavel/rotte eieren', 'Bakaliet/Fenol', 'Teer', 'Ammoniak'}
    PARTICULATE   = {'Zwart/grijs', 'Stofwolk'}
    if detail_str in COKE_SPECIFIC:
        return 'coke_specific'
    if detail_str in PARTICULATE:
        return 'particulate'
    return 'generic'


def score_citizen_reports(citizen_df, postcode, arrival, departure,
                          chimney_lat, chimney_lon):
    """
    Match citizen reports to a plume window and return structured evidence.

    The LLM receives categorical descriptor counts and distance context only.

    Returns dict with:
      n_reports     : total reports filed for this postcode
      n_valid       : reports falling within the plume arrival-departure window
      dist_m        : haversine distance from chimney to postcode centroid (m)
      detail_counts : {'coke_specific': int, 'particulate': int, 'generic': int}
      reports       : list of per-report dicts (datetime, detail, category,
                      minutes_after, dist_m)
    """
    pc_reports = citizen_df[
        citizen_df['postcode4'].astype(str) == str(postcode)
    ].copy()
    if pc_reports.empty:
        return {'n_reports': 0, 'n_valid': 0, 'dist_m': None,
                'detail_counts': {}, 'reports': []}

    cutoff_late = departure + pd.Timedelta(minutes=DISCARD_AFTER_MIN)
    valid = pc_reports[
        (pc_reports['datetime_waarneming'] >= arrival) &
        (pc_reports['datetime_waarneming'] <= cutoff_late)
    ].copy()

    if valid.empty:
        return {'n_reports': len(pc_reports), 'n_valid': 0, 'dist_m': None,
                'detail_counts': {}, 'reports': []}

    # Distance from chimney to this postcode centroid
    pc_geom = postcode_zones[
        postcode_zones['postcode'].astype(str) == str(postcode)
    ]
    if not pc_geom.empty and pd.notna(pc_geom.iloc[0]['centroid_lat']):
        dist_m = haversine_m(
            chimney_lat, chimney_lon,
            pc_geom.iloc[0]['centroid_lat'],
            pc_geom.iloc[0]['centroid_lon'],
        )
    else:
        dist_m = None

    scored        = []
    detail_counts = {}

    for _, row in valid.iterrows():
        mins   = (row['datetime_waarneming'] - arrival).total_seconds() / 60
        detail = str(row.get('soort_geur_geluid_stof', 'Anders'))
        cat    = _detail_category(detail)
        detail_counts[cat] = detail_counts.get(cat, 0) + 1
        scored.append({
            'datetime':      row['datetime_waarneming'],
            'detail':        detail,
            'category':      cat,
            'minutes_after': round(mins, 1),
            'dist_m':        round(dist_m) if dist_m is not None else None,
        })

    return {
        'n_reports':     len(pc_reports),
        'n_valid':       len(valid),
        'dist_m':        round(dist_m) if dist_m is not None else None,
        'detail_counts': detail_counts,
        'reports':       scored,
    }


In [ ]:
print(f"kgf_events shape: {kgf_events.shape}")
print(f"datetime range: {kgf_events['datetime_voorval'].min()} → {kgf_events['datetime_voorval'].max()}")
print(kgf_events['inrichting'].value_counts())

## 5 · Main aggregation loop

In [ ]:
def select_primary_monitor(chim_key, monitor_results):
    """
    Select the closest in-cone monitor as the primary AQ source.
    NL49573 is always included for KGF2 events regardless of wind direction
    due to its 321 m proximity (near-source fugitive capture).
    Returns (monitor_id, monitor_aq_dict).
    """
    candidates = {}
    for mon_id, res in monitor_results.items():
        dist = res.get('distance_m', float('inf'))

        if res.get('in_path', False):
            candidates[mon_id] = dist
            continue

        # Explicit proximity override: NL49573/KGF2 only
        if (chim_key, mon_id) in PROXIMITY_OVERRIDE:
            candidates[mon_id] = dist

    if not candidates:
        return None, {}
    primary = min(candidates, key=candidates.get)
    return primary, monitor_results.get(primary, {})


citizen_postcodes = set(citizen_reports['postcode4'].astype(str).unique())
event_records   = []
citizen_details = []

for i, (idx, event) in enumerate(kgf_events.iterrows()):
    if i % 50 == 0:
        print(f"Processing event {i}/{len(kgf_events)}...")

    event_time = event['datetime_voorval']
    chimney    = event['inrichting']
    chim_key   = 'KGF1' if 'KGF1' in chimney.upper() else 'KGF2'
    t_naive    = event_time.replace(tzinfo=None) \
                 if hasattr(event_time, 'tzinfo') else event_time

    # Weather
    weather = interpolate_weather(t_naive, weather_data)
    pg      = pasquill_class(weather, t_naive)

    # Monitor puff matching + AQ scoring
    monitor_results = {}
    for mon_id, mon_coords in MONITORS.items():
        puff = puff_times(chim_key, t_naive, weather,
                          mon_coords['lat'], mon_coords['lon'])
        if not puff['in_path']:
            monitor_results[mon_id] = {'in_path': False}
            # Still compute AQ for proximity override candidate
            if (chim_key, mon_id) in PROXIMITY_OVERRIDE:
                aq_z = get_aq_at_peak(mon_id, t_naive, aq_with_z.get(mon_id))
                monitor_results[mon_id] = {**puff, **aq_z,
                                           'in_path': False,
                                           'distance_m': haversine_m(
                                               CHIMNEYS[chim_key]['lat'],
                                               CHIMNEYS[chim_key]['lon'],
                                               mon_coords['lat'],
                                               mon_coords['lon'])}
            continue
        aq_z = get_aq_at_peak(mon_id, puff['peak'], aq_with_z.get(mon_id))
        monitor_results[mon_id] = {**puff, **aq_z}

    # Primary monitor + severity scores
    primary_mon, primary_aq = select_primary_monitor(chim_key, monitor_results)

    fp_z  = primary_aq.get('fingerprint_z', np.nan)
    gn_z  = primary_aq.get('general_z',     np.nan)
    sv_z  = primary_aq.get('severity_z',    np.nan)
    raw_c = primary_aq.get('raw_concentrations', {})

    # Concurrent attribution
    concurrent = kgf_events[
        (kgf_events.index != idx) &
        (kgf_events['datetime_voorval'].between(
            t_naive - pd.Timedelta(hours=3),
            t_naive + pd.Timedelta(hours=3),
        ))
    ]
    if concurrent.empty:
        con_share = 1.0
    else:
        this_z  = abs(sv_z) if not np.isnan(sv_z) else 0.5
        other_z = this_z * len(concurrent)
        con_share = (this_z / (this_z + other_z)
                     if (this_z + other_z) > 0
                     else 1 / (1 + len(concurrent)))

    adj_severity = sv_z * con_share if not np.isnan(sv_z) else np.nan

    # Offshore flag
    plume_dir = (weather.get('wind_dir_deg', 0) + 180) % 360
    offshore  = (plume_dir >= 235) or (plume_dir <= 5)

    # Citizen report matching
    chim_coords         = CHIMNEYS[chim_key]
    total_valid_reports = 0
    postcodes_in_path   = []
    agg_detail          = {'coke_specific': 0, 'particulate': 0, 'generic': 0}

    for _, pc_row in postcode_zones.iterrows():
        pc = str(pc_row['postcode'])
        if pc not in citizen_postcodes:
            continue
        if pd.isna(pc_row['centroid_lat']) or pd.isna(pc_row['centroid_lon']):
            continue

        puff_pc = puff_times(chim_key, t_naive, weather,
                             pc_row['centroid_lat'], pc_row['centroid_lon'],
                             target_geom=pc_row['geometry_shape'])
        if not puff_pc['in_path']:
            continue

        postcodes_in_path.append(pc)
        cs = score_citizen_reports(
            citizen_reports, pc,
            puff_pc['arrival'], puff_pc['departure'],
            chimney_lat=chim_coords['lat'],
            chimney_lon=chim_coords['lon'],
        )
        total_valid_reports += cs['n_valid']

        # Accumulate descriptor category counts across all postcodes
        for cat, n in cs['detail_counts'].items():
            agg_detail[cat] = agg_detail.get(cat, 0) + n

        for rep in cs['reports']:
            citizen_details.append({
                'event_idx':      idx,
                'event_datetime': event_time,
                'chimney':        chim_key,
                'postcode':       pc,
                'puff_arrival':   puff_pc['arrival'],
                'puff_departure': puff_pc['departure'],
                'dist_m':         cs['dist_m'],
                **rep,
                'con_share':      con_share,
            })

    # Append event record
    event_records.append({
        'event_idx':               idx,
        'chimney':                 chim_key,
        'event_datetime':          event_time,
        'soort_voorval':           event.get('soort_voorval', ''),
        'wind_dir_deg':            weather.get('wind_dir_deg'),
        'wind_speed_ms':           weather.get('wind_speed_ms'),
        'temp_c':                  weather.get('temp_c'),
        'radiation_jcm2':          weather.get('radiation_jcm2'),
        'cloud_cover_n':           weather.get('N'),
        'humidity_pct':            weather.get('U'),
        'pg_class':                pg,
        'plume_direction_deg':     plume_dir,
        'offshore_flag':           offshore,
        'postcodes_in_path':       '|'.join(sorted(postcodes_in_path)),
        'n_postcodes_in_path':     len(postcodes_in_path),
        'primary_monitor':         primary_mon,
        # AQ severity scores
        'aq_fingerprint_z':        fp_z,
        'aq_general_z':            gn_z,
        'aq_severity_z':           adj_severity,
        'aq_composite_z':          adj_severity, # alias for further work (bad naming practice)
        'aq_composite_z_raw':      sv_z,
        # Individual z-scores + raw concentrations from primary monitor
        **{f'z_{c}':   primary_aq.get(c,   np.nan) for c in TARGET_CHEMICALS},
        **{f'raw_{c}': raw_c.get(c, np.nan)         for c in TARGET_CHEMICALS},
        # Citizen descriptor counts (categorical, no numerical score passed to LLM)
        'citizen_coke_specific':   agg_detail['coke_specific'],
        'citizen_particulate':     agg_detail['particulate'],
        'citizen_generic':         agg_detail['generic'],
        'concurrent_share':        con_share,
        'n_valid_citizen_reports': total_valid_reports,
    })

df_events   = pd.DataFrame(event_records)
df_citizens = pd.DataFrame(citizen_details) if citizen_details else pd.DataFrame()

print(f"Aggregation complete.")
print(f"  Events processed           : {len(df_events)}")
print(f"  Citizen report matches     : {len(df_citizens)}")
print(f"  Offshore events            : {df_events['offshore_flag'].sum()}")
print(f"  Events with AQ data        : {df_events['aq_severity_z'].notna().sum()}")
print(f"  Events with citizen reports: {(df_events['n_valid_citizen_reports']>0).sum()}")
print(f"  NL49573 assigned           : {(df_events['primary_monitor']=='NL49573').sum()}")


## 6 · Save structured datasets

In [ ]:
df_events.to_parquet(PROCESSED_DIR / "aggregated_events.parquet", index=False)
if not df_citizens.empty:
    df_citizens.to_parquet(PROCESSED_DIR / "matched_citizen_reports.parquet", index=False)

print("Saved:")
print(f"  aggregated_events.parquet       ({len(df_events)} rows × {len(df_events.columns)} cols)")
print(f"  matched_citizen_reports.parquet ({len(df_citizens)} rows)")
print(f"\nColumns in aggregated_events:")
for col in df_events.columns:
    print(f"  {col}")

## 7 · Natural language paragraph generation

In [ ]:
COMPASS = ['N','NNE','NE','ENE','E','ESE','SE','SSE',
           'S','SSW','SW','WSW','W','WNW','NW','NNW']
 
 
def format_paragraph(row, df_cit):
    chim  = row['chimney']
    t     = pd.Timestamp(row['event_datetime']).strftime('%d %B %Y at %H:%M')
    pg    = str(row['pg_class'])
    wdir  = row['wind_dir_deg']
    wspd  = row['wind_speed_ms']
    pdir  = row['plume_direction_deg']
    wcomp = COMPASS[int((wdir + 11.25) / 22.5) % 16] if pd.notna(wdir) else '?'
    pg_desc = PG_PLAIN.get(pg, f'class {pg}')
 
    # 1. Header
    header = (
        f"Emission event at {chim} reported on {t}. "
        f"Event type: {row.get('soort_voorval', 'unspecified')}."
    )
 
    # 2. Atmospheric context
    offshore_note = (
        " Wind conditions directed the plume offshore; "
        "community exposure from residential postcodes is unlikely."
        if row['offshore_flag'] else ""
    )
 
    con_share = float(row['concurrent_share'])
    if con_share < 0.99:
        n_concurrent = max(round(1 / con_share) - 1, 1)
        concurrent_note = (
            f" Note: {n_concurrent} concurrent KGF event(s) within ±3 hours; "
            f"AQ readings attributed {round(con_share*100)}% to this event."
        )
    else:
        concurrent_note = ""
 
    atm = (
        f"Wind {wcomp} at {wspd:.1f} m/s. "
        f"Atmospheric stability: Pasquill-Gifford class {pg} ({pg_desc}). "
        f"Estimated plume direction: {pdir:.0f}°."
        f"{offshore_note}{concurrent_note}"
    )
 
    pm   = row.get('primary_monitor')
    fp_z = row.get('aq_fingerprint_z')
    gn_z = row.get('aq_general_z')
    sv_z = row.get('aq_severity_z')
 
    if pd.notna(sv_z) and pm:
        # Individual chemical z-scores + raw concentrations
        chem_parts = []
        for chem in COKE_FINGERPRINT_CHEMS + GENERAL_COMBUSTION_CHEMS + ['NO2']:
            z_val   = row.get(f'z_{chem}')
            raw_val = row.get(f'raw_{chem}')
            if pd.notna(z_val):
                raw_str = f" ({raw_val:.2f} µg/m³)" if pd.notna(raw_val) else ""
                chem_parts.append(f"{chem}: z={z_val:.2f}{raw_str}")
 
        chem_detail = '; '.join(chem_parts) if chem_parts else 'no readings available'
 
        fp_str = (f"Coke fingerprint group (SO2, H2S, NO, NOx): "
                  f"max z={fp_z:.2f}" if pd.notna(fp_z)
                  else "Coke fingerprint group: no data")
        gn_str = (f"General combustion group (BC, PM10, PM2.5, CO): "
                  f"max z={gn_z:.2f}" if pd.notna(gn_z)
                  else "General combustion group: no data")
 
        aq_text = (
            f"Monitor {pm} (primary, in plume cone): {chem_detail}. "
            f"{fp_str}. {gn_str}. "
            f"Primary severity indicator: z={sv_z:.2f}. "
            f"Values are 1-hour means; instantaneous peak during short-duration "
            f"puff events may be higher."
        )
 
    elif pm:
        # Particulate monitor (PM10 and PM2.5 only — NL49557/NL49570/NL49572/NL49573)
        pm10_z   = row.get('z_PM10')
        pm25_z   = row.get('z_PM2.5')
        pm10_raw = row.get('raw_PM10')
        pm25_raw = row.get('raw_PM2.5')
 
        parts = []
        if pd.notna(pm10_z):
            raw_str = f" ({pm10_raw:.2f} µg/m³)" if pd.notna(pm10_raw) else ""
            parts.append(f"PM10: z={pm10_z:.2f}{raw_str}")
        if pd.notna(pm25_z):
            raw_str = f" ({pm25_raw:.2f} µg/m³)" if pd.notna(pm25_raw) else ""
            parts.append(f"PM2.5: z={pm25_z:.2f}{raw_str}")
 
        readings = "; ".join(parts) if parts else "no readings available"
        aq_text = (
            f"Monitor {pm} (particulate monitor — PM10 and PM2.5 only, in plume cone): "
            f"{readings}. "
            f"Gas-phase coke fingerprint chemicals (SO2, H2S, NO, NOx) are not "
            f"measured at this station. Severity indicator reflects particulate "
            f"signal only."
        )
    else:
        aq_text = (
            f"No monitor fell within the plume cone for this event. "
            f"AQ evidence absent."
        )
 
    # 4. Citizen report summary
    nv     = int(row['n_valid_citizen_reports'])
    n_coke = int(row.get('citizen_coke_specific', 0))
    n_part = int(row.get('citizen_particulate', 0))
    n_gen  = int(row.get('citizen_generic', 0))
 
    if nv > 0:
        pcs    = row['postcodes_in_path'].split('|') if row['postcodes_in_path'] else []
        ev_cit = df_cit[df_cit['event_idx'] == row['event_idx']]
 
        dists = ev_cit['dist_m'].dropna() if 'dist_m' in ev_cit.columns else pd.Series(dtype=float)
        dist_str = (
            f"Distance range of reporting postcodes: "
            f"{dists.min()/1000:.1f} km to {dists.max()/1000:.1f} km from chimney. "
            if not dists.empty else ""
        )
 
        cit_text = (
            f"{nv} citizen report(s) within plume window across {len(pcs)} postcode(s). "
            f"Coke-specific descriptors (Kooks, Zwavel/rotte eieren, Bakeliet/Fenol, "
            f"Teer, Ammoniak): {n_coke}. "
            f"Particulate descriptors (Zwart/grijs, Stofwolk): {n_part}. "
            f"Generic descriptors: {n_gen}. "
            f"{dist_str}"
        )
    elif row['offshore_flag']:
        cit_text = "No citizen reports expected (plume offshore)."
    elif row['n_postcodes_in_path'] == 0:
        cit_text = "No residential postcodes in plume path."
    else:
        cit_text = (
            f"No citizen reports within valid window "
            f"({row['n_postcodes_in_path']} postcode(s) in path). "
            "Possible low-visibility, nocturnal, or short-duration release."
        )
 
    para = f"{header} {atm} {aq_text} {cit_text}"
 
    struct = {
        'event_id':              int(row['event_idx']),
        'chimney':               chim,
        'datetime':              str(row['event_datetime']),
        'soort_voorval':         row.get('soort_voorval', ''),
        'pg_class':              pg,
        'pg_description':        pg_desc,
        'wind_dir_deg':          round(float(wdir), 1) if pd.notna(wdir) else None,
        'wind_speed_ms':         round(float(wspd), 2) if pd.notna(wspd) else None,
        'plume_dir_deg':         round(float(pdir), 1),
        'offshore':              bool(row['offshore_flag']),
        'primary_monitor':       pm,
        'aq_fingerprint_z':      round(float(fp_z), 3) if pd.notna(fp_z) else None,
        'aq_general_z':          round(float(gn_z), 3) if pd.notna(gn_z) else None,
        'aq_severity_z':         round(float(sv_z), 3) if pd.notna(sv_z) else None,
        'aq_per_chemical_z':     {
            c: round(float(row[f'z_{c}']), 3)
            for c in TARGET_CHEMICALS if pd.notna(row.get(f'z_{c}'))
        },
        'raw_concentrations_ugm3': {
            c: round(float(row[f'raw_{c}']), 3)
            for c in TARGET_CHEMICALS if pd.notna(row.get(f'raw_{c}'))
        },
        'concurrent_share':       round(float(con_share), 3),
        'n_citizen_reports':      nv,
        'citizen_coke_specific':  n_coke,
        'citizen_particulate':    n_part,
        'citizen_generic':        n_gen,
        'postcodes_in_path':      row['postcodes_in_path'].split('|')
                                  if row['postcodes_in_path'] else [],
    }
    return para, struct

In [ ]:
paragraphs = []
for _, row in df_events.iterrows():
    para, struct = format_paragraph(row, df_citizens)
    paragraphs.append({
        'event_idx':       row['event_idx'],
        'chimney':         row['chimney'],
        'datetime':        row['event_datetime'],
        'paragraph':       para,
        'retrieval_query': para,
        'structured_json': json.dumps(struct, default=str),
    })

df_paragraphs = pd.DataFrame(paragraphs)
df_paragraphs.to_parquet(PROCESSED_DIR / "event_paragraphs.parquet", index=False)
print(f"Saved event_paragraphs.parquet ({len(df_paragraphs)} events)")
print(df_paragraphs.head(5))

## 8 · Query interface
Filter events by date range and/or chimney for downstream stages.

In [ ]:
def query_events(date_from, date_to, chimney=None):
    """
    Retrieve aggregated event rows and paragraphs for a given period.

    Parameters
    ----------
    date_from : str  e.g. '2025-03-01'
    date_to   : str  e.g. '2025-03-31'
    chimney   : str or None  — 'KGF1', 'KGF2', or None for both

    Returns
    -------
    events     : DataFrame
    paragraphs : DataFrame (paragraph + structured_json per event)
    """
    mask = (
        (df_events['event_datetime'] >= pd.Timestamp(date_from)) &
        (df_events['event_datetime'] <= pd.Timestamp(date_to))
    )
    if chimney:
        mask &= df_events['chimney'] == chimney

    ev  = df_events[mask].copy()
    par = df_paragraphs[df_paragraphs['event_idx'].isin(ev['event_idx'])].copy()
    print(f"Events: {len(ev)}  |  period: {date_from} → {date_to}"
          + (f"  |  chimney: {chimney}" if chimney else ""))
    return ev, par